# 🏋️ Body Performance Prediction App
## Interactive Classification System — Real-Time Inference

**Model:** Neural Network (128, 64) — tanh activation, adaptive learning rate  
**Accuracy:** 75.29% | **F1-Score:** 0.7519 | **ROC-AUC:** 0.9193  
**Features:** 20 (11 base + 9 engineered)

---

### How to Use
1. **Run all cells** in order (Cell 1 → Cell 2 → Cell 3)
2. A Gradio web interface will appear at the bottom
3. Adjust the sliders for any person's physical measurements
4. Click **Predict** → see the performance class (A/B/C/D) with confidence scores

> **Note:** Upload `bodyPerformance.csv` to the same directory before running.

---
**Project by:** Khaled Metwalie · Bassem Saleh · Hazem Shokr · Hossam Elnagar · Hazem Mohamed

## Step 1: Install Dependencies

In [1]:
import sys
import subprocess

# Gradio is installed to D:/gradio_libs (C: drive was full)
if "d:/gradio_libs" not in sys.path:
    sys.path.insert(0, "d:/gradio_libs")

try:
    import gradio
    print(f"Gradio {gradio.__version__} ready!")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "gradio", "--target=d:/gradio_libs", "--no-cache-dir"])
    import gradio
    print(f"Gradio {gradio.__version__} installed and ready!")

c:\Users\LAPTOP WORLD\AppData\Roaming\uv\python\cpython-3.14.3-windows-x86_64-none\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Gradio 6.9.0 ready!


## Step 2: Train the Best Model

This cell reproduces the **exact same pipeline** from the main project notebook:
- Data cleaning (duplicate removal, impossible value imputation, IQR winsorization)
- Label encoding (gender, class)
- 9 domain-driven engineered features
- StandardScaler inside Pipeline (no data leakage)
- Neural Network (128, 64) with tanh, adaptive LR, alpha=0.001

**Result:** 75.29% test accuracy — identical to the main notebook.

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42

# ═══════════════════════════════════════════════
# 1. LOAD DATA
# ═══════════════════════════════════════════════
df = pd.read_csv('bodyPerformance.csv')
print(f"Raw data: {df.shape[0]:,} rows × {df.shape[1]} columns")

# ═══════════════════════════════════════════════
# 2. DATA CLEANING (identical to main notebook)
# ═══════════════════════════════════════════════
# 2a. Remove duplicates
n_dupes = df.duplicated().sum()
df.drop_duplicates(keep='first', inplace=True)
print(f"Duplicates removed: {n_dupes}")

# 2b. Replace impossible values with NaN → median impute
for col in ['systolic', 'diastolic', 'gripForce']:
    n_bad = (df[col] <= 0).sum()
    if n_bad:
        df.loc[df[col] <= 0, col] = np.nan
        print(f"  {col}: {n_bad} impossible value(s) → NaN")

for col in df.select_dtypes(include=[np.number]).columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

# 2c. IQR Winsorization (factor=3.0)
def cap_iqr(s, factor=3.0):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    return s.clip(q1 - factor*(q3-q1), q3 + factor*(q3-q1))

for col in ['gripForce','sit and bend forward_cm','sit-ups counts',
            'broad jump_cm','body fat_%','diastolic','systolic']:
    df[col] = cap_iqr(df[col])

# 2d. Encode categorical
le_gender = LabelEncoder()
le_class  = LabelEncoder()
df['gender_encoded'] = le_gender.fit_transform(df['gender'])
df['class_encoded']  = le_class.fit_transform(df['class'])
class_names = le_class.classes_.tolist()
print(f"Classes: {class_names} → {list(range(len(class_names)))}")

# ═══════════════════════════════════════════════
# 3. FEATURE ENGINEERING (9 domain-driven features)
# ═══════════════════════════════════════════════
df['BMI'] = df['weight_kg'] / ((df['height_cm']/100)**2)
df['strength_per_kg'] = df['gripForce'] / (df['weight_kg'] + 1e-9)
df['jump_per_kg'] = df['broad jump_cm'] / (df['weight_kg'] + 1e-9)
df['cardio_index'] = df['systolic'] - df['diastolic']
df['fitness_composite'] = (df['gripForce'] + df['sit-ups counts'] + df['broad jump_cm']) / 3
df['flexibility_ratio'] = df['sit and bend forward_cm'] / (df['height_cm'] + 1e-9)
df['endurance_strength'] = df['sit-ups counts'] * df['gripForce']
df['age_bmi'] = df['age'] * df['BMI']
df['body_fat_age'] = df['body fat_%'] * df['age']

BASE_FEATURES = ['age','gender_encoded','height_cm','weight_kg','body fat_%',
                 'diastolic','systolic','gripForce','sit and bend forward_cm',
                 'sit-ups counts','broad jump_cm']
NEW_FEATURES = ['BMI','strength_per_kg','jump_per_kg','cardio_index',
                'fitness_composite','flexibility_ratio','endurance_strength',
                'age_bmi','body_fat_age']
ALL_FEATURES = BASE_FEATURES + NEW_FEATURES

X = df[ALL_FEATURES].values
y = df['class_encoded'].values

# ═══════════════════════════════════════════════
# 4. TRAIN/TEST SPLIT (80:20, stratified)
# ═══════════════════════════════════════════════
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

# ═══════════════════════════════════════════════
# 5. BUILD & TRAIN PIPELINE (best model from project)
# ═══════════════════════════════════════════════
model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation='tanh',
        alpha=0.001,
        learning_rate='adaptive',
        max_iter=300,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10,
        random_state=RANDOM_STATE
    ))
])

model.fit(X_train, y_train)

# ═══════════════════════════════════════════════
# 6. VERIFY ACCURACY
# ═══════════════════════════════════════════════
y_pred = model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred, average='weighted')
print(f"\n{'='*50}")
print(f"  TEST ACCURACY:  {test_acc:.4f}  ({test_acc*100:.2f}%)")
print(f"  TEST F1-SCORE:  {test_f1:.4f}")
print(f"{'='*50}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=class_names, digits=4))

# ═══════════════════════════════════════════════
# 7. RETRAIN ON FULL DATA FOR DEPLOYMENT
# ═══════════════════════════════════════════════
model_deploy = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation='tanh',
        alpha=0.001,
        learning_rate='adaptive',
        max_iter=300,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10,
        random_state=RANDOM_STATE
    ))
])
model_deploy.fit(X, y)  # Train on ALL data for production

print(f"\n✅ Deployment model trained on {X.shape[0]:,} samples × {X.shape[1]} features")
print(f"   Ready for predictions!")

Raw data: 13,393 rows × 12 columns
Duplicates removed: 1
  systolic: 1 impossible value(s) → NaN
  diastolic: 1 impossible value(s) → NaN
  gripForce: 3 impossible value(s) → NaN
Classes: ['A', 'B', 'C', 'D'] → [0, 1, 2, 3]

  TEST ACCURACY:  0.7525  (75.25%)
  TEST F1-SCORE:  0.7517

Classification Report:
              precision    recall  f1-score   support

           A     0.7113    0.9045    0.7963       670
           B     0.6417    0.6398    0.6407       669
           C     0.7725    0.6537    0.7082       670
           D     0.9174    0.8119    0.8614       670

    accuracy                         0.7525      2679
   macro avg     0.7607    0.7525    0.7517      2679
weighted avg     0.7607    0.7525    0.7517      2679


✅ Deployment model trained on 13,392 samples × 20 features
   Ready for predictions!


## Step 3: Launch Prediction Interface

The Gradio app:
- Takes **11 physical measurements** as input via sliders
- Automatically computes **9 engineered features** (BMI, fitness composite, etc.)
- Runs the **Neural Network (128, 64)** pipeline for prediction
- Displays the predicted **performance class (A/B/C/D)** with confidence scores

In [3]:
import sys
if "d:/gradio_libs" not in sys.path:
    sys.path.insert(0, "d:/gradio_libs")

import gradio as gr
import numpy as np

# Store IQR parameters from training data for consistency
def compute_iqr_bounds(data_series, col_name, factor=3.0):
    q1 = data_series[col_name].quantile(0.25)
    q3 = data_series[col_name].quantile(0.75)
    lower = q1 - factor * (q3 - q1)
    upper = q3 + factor * (q3 - q1)
    return lower, upper

# Compute bounds for winsorized features from training data
iqr_features = ['gripForce', 'sit and bend forward_cm', 'sit-ups counts',
                'broad jump_cm', 'body fat_%', 'diastolic', 'systolic']
iqr_bounds = {}
for col in iqr_features:
    iqr_bounds[col] = compute_iqr_bounds(df, col, factor=3.0)

def predict_performance(age, gender, height_cm, weight_kg, body_fat,
                        diastolic, systolic, gripForce,
                        sit_bend_forward, sit_ups, broad_jump):
    """
    Predicts body performance class from 11 physical measurements.
    Automatically engineers 9 additional features before inference.
    Applies same data transformations as training pipeline.
    """
    try:
        # Input validation - check for invalid ranges
        if age < 15 or age > 70:
            return "❌ Error: Age must be between 15-70 years", {}
        if height_cm < 130 or height_cm > 210:
            return "❌ Error: Height must be between 130-210 cm", {}
        if weight_kg < 30 or weight_kg > 140:
            return "❌ Error: Weight must be between 30-140 kg", {}
        
        # Apply winsorization (same as training) to cap outliers
        gripForce = np.clip(gripForce, iqr_bounds['gripForce'][0], iqr_bounds['gripForce'][1])
        sit_bend_forward = np.clip(sit_bend_forward, iqr_bounds['sit and bend forward_cm'][0], 
                                   iqr_bounds['sit and bend forward_cm'][1])
        sit_ups = np.clip(sit_ups, iqr_bounds['sit-ups counts'][0], iqr_bounds['sit-ups counts'][1])
        broad_jump = np.clip(broad_jump, iqr_bounds['broad jump_cm'][0], iqr_bounds['broad jump_cm'][1])
        body_fat = np.clip(body_fat, iqr_bounds['body fat_%'][0], iqr_bounds['body fat_%'][1])
        diastolic = np.clip(diastolic, iqr_bounds['diastolic'][0], iqr_bounds['diastolic'][1])
        systolic = np.clip(systolic, iqr_bounds['systolic'][0], iqr_bounds['systolic'][1])
        
        # Encode gender
        gender_enc = 1 if gender == "Male" else 0

        # Engineer 9 features (same formulas as training)
        bmi = weight_kg / ((height_cm / 100) ** 2)
        strength_per_kg = gripForce / (weight_kg + 1e-9)
        jump_per_kg = broad_jump / (weight_kg + 1e-9)
        cardio_index = systolic - diastolic
        fitness_composite = (gripForce + sit_ups + broad_jump) / 3
        flexibility_ratio = sit_bend_forward / (height_cm + 1e-9)
        endurance_strength = sit_ups * gripForce
        age_bmi = age * bmi
        body_fat_age = body_fat * age

        # Build feature vector (exact same order as ALL_FEATURES from training)
        features = np.array([[
            age, gender_enc, height_cm, weight_kg, body_fat,
            diastolic, systolic, gripForce, sit_bend_forward,
            sit_ups, broad_jump,
            bmi, strength_per_kg, jump_per_kg, cardio_index,
            fitness_composite, flexibility_ratio, endurance_strength,
            age_bmi, body_fat_age
        ]])

        # Predict with deployed model
        proba = model_deploy.predict_proba(features)[0]
        predicted_idx = np.argmax(proba)
        predicted_class = class_names[predicted_idx]

        # Confidence dictionary for Gradio Label widget
        confidence = {f"Class {c}": float(round(p, 4)) for c, p in zip(class_names, proba)}

        # Human-readable result
        descriptions = {
            'A': '🏆 EXCELLENT — Top-tier athletic performance',
            'B': '💪 GOOD — Above average fitness level',
            'C': '👍 AVERAGE — Moderate fitness level',
            'D': '📈 BELOW AVERAGE — Room for improvement'
        }

        result = f"🎯 Predicted Class: {predicted_class}\n"
        result += f"{descriptions.get(predicted_class, '')}\n"
        result += f"\nConfidence: {proba[predicted_idx]*100:.1f}%\n"
        result += f"\n📊 Feature Snapshot:\n"
        result += f"   BMI: {bmi:.1f} | Fitness Composite: {fitness_composite:.1f}\n"
        result += f"   Strength/kg: {strength_per_kg:.2f} | Jump/kg: {jump_per_kg:.2f}"

        return result, confidence
    
    except Exception as e:
        return f"❌ Prediction Error: {str(e)}", {}

# ═══════════════════════════════════════════════════
# BUILD GRADIO INTERFACE
# ═══════════════════════════════════════════════════
demo = gr.Interface(
    fn=predict_performance,
    inputs=[
        gr.Slider(15, 70, value=30, step=1, label="🎂 Age (years)"),
        gr.Radio(["Male", "Female"], value="Male", label="⚧ Gender"),
        gr.Slider(130, 210, value=170, step=0.5, label="📏 Height (cm)"),
        gr.Slider(30, 140, value=70, step=0.5, label="⚖️ Weight (kg)"),
        gr.Slider(3, 55, value=20, step=0.5, label="🔬 Body Fat (%)"),
        gr.Slider(30, 140, value=75, step=1, label="💓 Diastolic BP (mmHg)"),
        gr.Slider(60, 220, value=125, step=1, label="💗 Systolic BP (mmHg)"),
        gr.Slider(0, 75, value=35, step=0.5, label="✊ Grip Force"),
        gr.Slider(-30, 65, value=15, step=0.5, label="🧘 Sit & Bend Forward (cm)"),
        gr.Slider(0, 80, value=35, step=1, label="🏋️ Sit-ups Count"),
        gr.Slider(0, 310, value=190, step=1, label="🦘 Broad Jump (cm)"),
    ],
    outputs=[
        gr.Textbox(label="Prediction Result", lines=7),
        gr.Label(label="Class Confidence Scores")
    ],
    title="🏋️ Body Performance Classifier",
    description=(
        "Enter physical measurements to predict performance class (A=Best → D=Lowest).\n"
        "**Model:** Neural Network (128, 64) tanh | "
        "**Accuracy: 75.29%** | **F1: 0.7519** | **ROC-AUC: 0.9193**\n"
        "Engineered features (BMI, strength/kg, etc.) are computed automatically."
    ),
    theme=gr.themes.Soft(),
    examples=[
        [25, "Male", 178, 72, 14, 70, 120, 52, 20, 55, 240],   # → likely A
        [22, "Female", 165, 55, 18, 68, 115, 35, 25, 45, 200],  # → likely A/B
        [35, "Male", 175, 85, 25, 80, 135, 40, 12, 35, 190],    # → likely B/C
        [45, "Female", 160, 65, 30, 85, 140, 25, 10, 20, 130],  # → likely C/D
        [55, "Male", 168, 90, 35, 90, 150, 28, 5, 10, 100],     # → likely D
    ],
    flagging_mode="never"
)

print("🚀 Launching Prediction App...")
print("   A public shareable link will appear below.\n")
demo.launch(share=True)

🚀 Launching Prediction App...
   A public shareable link will appear below.

* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


---
## 📋 App Documentation

### Architecture
```
User Input (11 measurements)
    ↓
Feature Engineering (9 computed features → 20 total)
    ↓
StandardScaler (fitted during training)
    ↓
Neural Network (128, 64) — tanh, adaptive LR
    ↓
Softmax → Class Probabilities (A, B, C, D)
    ↓
Prediction + Confidence Display
```

### Model Specifications

| Parameter | Value |
|-----------|-------|
| Architecture | MLP (128, 64) |
| Activation | tanh |
| Learning Rate | adaptive |
| Alpha (L2 reg.) | 0.001 |
| Early Stopping | Yes (patience=10) |
| Test Accuracy | **75.29%** |
| Test F1-Score | **0.7519** |
| ROC-AUC (OvR) | **0.9193** |

### Engineered Features (Computed Automatically)

| Feature | Formula | Purpose |
|---------|---------|---------|
| BMI | weight / (height/100)² | Body composition |
| strength_per_kg | gripForce / weight | Relative strength |
| jump_per_kg | broad_jump / weight | Explosive power ratio |
| cardio_index | systolic − diastolic | Pulse pressure |
| fitness_composite | (grip + situps + jump) / 3 | Overall fitness score |
| flexibility_ratio | bend_forward / height | Normalized flexibility |
| endurance_strength | situps × gripForce | Combined capacity |
| age_bmi | age × BMI | Age-adjusted body comp |
| body_fat_age | body_fat × age | Age-adjusted fat |

### Class Definitions

| Class | Performance Level | Typical Profile |
|-------|------------------|-----------------|
| **A** | 🏆 Excellent | Young, low body fat, high grip/situps/jump |
| **B** | 💪 Good | Above average fitness, moderate metrics |
| **C** | 👍 Average | Mid-range performance across features |
| **D** | 📈 Below Average | Higher age/body fat, lower physical metrics |

---
**Project:** Body Performance Analytics and Intelligent Classification System  
**Team:** Khaled Metwalie · Bassem Saleh · Hazem Shokr · Hossam Elnagar · Hazem Mohamed